<a href="https://colab.research.google.com/github/adrienyldefonso-sys/Proyecto-Integrador/blob/main/entrenamiento_papas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile

# Descomprimir las carpetas en el directorio 'dataset'
with zipfile.ZipFile('/potato_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

print("¡Dataset descomprimido con éxito!")

¡Dataset descomprimido con éxito!


In [ ]:
import tensorflow as tf

# Parámetros básicos
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
DATA_DIR = 'dataset'

# 1. Cargar el 80% para Entrenamiento
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 2. Cargar el 20% para Validación
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Confirmar las clases encontradas
print("Clases detectadas:", train_ds.class_names)

Found 2152 files belonging to 3 classes.
Using 1722 files for training.
Found 2152 files belonging to 3 classes.
Using 430 files for validation.
Clases detectadas: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [ ]:
import tensorflow as tf

# 1. Cargar el modelo base MobileNetV2 preentrenado en ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='imagenet'
)

# Congelar los pesos del modelo base para no reentrenar todas sus capas
base_model.trainable = False

# 2. Construir la red agregando las nuevas capas para tus 3 clases
inputs = tf.keras.Input(shape=(256, 256, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # Ajusta los píxeles para MobileNetV2
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)  # Ayuda a evitar el sobreajuste
outputs = tf.keras.layers.Dense(3, activation='softmax')(x)  # 3 clases (Early blight, Late blight, Healthy)

model = tf.keras.Model(inputs, outputs)

# 3. Compilar el modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Iniciar el entrenamiento (10 épocas)
EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

/tmp/ipykernel_1947/3581922346.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 59s 700ms/step - accuracy: 0.8368 - loss: 0.4400 - val_accuracy: 0.9279 - val_loss: 0.2230
Epoch 2/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.9483 - loss: 0.1663 - val_accuracy: 0.9605 - val_loss: 0.1507
Epoch 3/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9628 - loss: 0.1198 - val_accuracy: 0.9651 - val_loss: 0.1203
Epoch 4/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9704 - loss: 0.0968 - val_accuracy: 0.9674 - val_loss: 0.1071
Epoch 5/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9739 - loss: 0.0825 - val_accuracy: 0.9698 - val_loss: 0.0921
Epoch 6/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9803 - loss: 0.0687 - val_accuracy: 0.9744 - val_loss: 0.0814
Epoch 7/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9820 - loss: 0.0637 - val_accuracy: 0.9744 - val_loss: 0.0755
Epoch 8/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accurac

In [ ]:
import tensorflow as tf
from google.colab import files

# ==========================================
# PASO 5: Validar la precisión del modelo
# ==========================================
val_loss, val_acc = model.evaluate(val_ds)
print("\n-------------------------------------------")
print(f"Pérdida en validación (Loss): {val_loss:.4f}")
print(f"Precisión en validación (Accuracy): {val_acc * 100:.2f}%")
print("-------------------------------------------\n")

if val_acc >= 0.80:
    print("¡Validación exitosa! La precisión está por encima del 80% requisitado.")
else:
    print("La precisión es baja, considera entrenar por más épocas.")


# ==========================================
# PASO 6: Exportar a TensorFlow Lite (.tflite)
# ==========================================
# 1. Convertir el modelo a TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 2. Guardar el archivo .tflite en el entorno
tflite_filename = 'model.tflite'
with open(tflite_filename, 'wb') as f:
    f.write(tflite_model)

print(f"\n¡Modelo exportado con éxito como '{tflite_filename}'!")

# 3. Guardar las etiquetas de las clases en un archivo labels.txt (muy útil para Android)
labels = train_ds.class_names
with open('labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")

print("¡Archivo 'labels.txt' creado con los nombres de las clases!")

# 4. Descargar automáticamente los archivos a tu computadora
files.download(tflite_filename)
files.download('labels.txt')

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9814 - loss: 0.0601

-------------------------------------------
Pérdida en validación (Loss): 0.0601
Precisión en validación (Accuracy): 98.14%
-------------------------------------------

¡Validación exitosa! La precisión está por encima del 80% requisitado.
Saved artifact at '/tmp/tmpqq11fq4p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135708969893712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135708944880016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135708944880592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135711302664848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135708944880784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135708944881168: TensorSpec(shape=(), dtype=t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>